# Transcrição de Vídeo (.mkv) — Google Colab

Pipeline completo de transcrição fiel usando:

- **`faster-whisper`** rodando na **GPU T4 gratuita** do Colab (10-30× mais rápido que CPU local)
- **OpenRouter** (opcional) para revisão de fidelidade via LLM

O código é o mesmo do projeto local — este notebook só monta o ambiente, recebe o vídeo e dispara o pipeline.

## 1. Verificar GPU

Confirma que o Colab alocou GPU. Se aparecer `Sem GPU`, vá em **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!nvidia-smi -L
import torch
print(f"\nGPU disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    # Compat: torch >= 2.0 usa 'total_memory'; versões antigas usavam 'total_mem'
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")
else:
    print("⚠ Sem GPU. Vá em Ambiente de execução > Alterar tipo > T4 GPU")

GPU 0: Tesla T4 (UUID: GPU-d00b67b4-e3be-1d3b-9ff9-10b7ae9bdb9e)

GPU disponível: True
Device: Tesla T4
VRAM: 15.6 GB


## 2. Instalar `ffmpeg` e `uv`

Colab já tem ffmpeg, mas garantimos a versão. `uv` é o gerenciador de ambiente do projeto.

In [3]:
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null
!pip install -q uv
!ffmpeg -version | head -1
!uv --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 95.6 MB/s eta 0:00:00
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
uv 0.12.9 (x86_64-unknown-linux-gnu)


## 3. Obter o código do projeto

**Escolha UM dos caminhos abaixo** (rode só um):

- **Caminho A** — `git clone` (recomendado se o projeto está no GitHub/GitLab)
- **Caminho B** — Upload manual (zip do projeto sem `.venv/`)

> Se for repositório **privado**, prefira o Caminho B ou use um token na URL.

In [4]:
# Caminho A: git clone (recomendado)
#
# Estratégia: vai pra /content, clona o repo como `repo`, entra nele.
# Se /content/repo já existir (de um re-run), remove antes pra evitar erro.

import os, shutil
from pathlib import Path

REPO_URL = "https://github.com/AndySouzaDS/transcrever-videos.git"  # ← edite aqui

target = Path("/content/repo")
if target.exists():
    if (target / ".git").exists():
        print(f"ℹ /content/repo já existe e é um clone de {REPO_URL} — reaproveitando.")
        print("  (delete a pasta /content/repo manualmente se quiser clonar do zero)")
    else:
        raise RuntimeError(
            f"/content/repo existe mas não é um clone do projeto. "
            f"Remova manualmente: !rm -rf /content/repo"
        )
else:
    %cd /content
    !git clone $REPO_URL repo

%cd /content/repo
!ls

/content
Cloning into 'repo'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 47 (delta 14), reused 42 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 78.85 KiB | 9.86 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/repo
notebooks  pyproject.toml  README.md  src  uv.lock


In [ ]:
# Caminho B: upload manual (rode SÓ se NÃO usou o Caminho A)
#
# Use este caminho APENAS se:
#   - O Caminho A falhou (ex.: repo privado sem token), OU
#   - Você quer testar uma versão local antes de subir pro GitHub.
#
# O Caminho A (git clone) é o caminho padrão e recomendado.

# from google.colab import files
# import os
# os.makedirs("/content/repo", exist_ok=True)
# %cd /content/repo
# print("Faça upload de um .zip do projeto (sem .venv/, sem saidas/):")
# print("  No seu PC: zip -r projeto.zip . -x '.venv/*' '.git/*' 'saidas/*'")
# uploaded = files.upload()
# for nome in uploaded:
#     !unzip -q "$nome"
#     !rm "$nome"
# %cd /content

## 4. Instalar dependências do projeto

In [5]:
%%bash
# %%bash preserva o cwd dentro da célula (evita o bug do `!` + pipe que
# reseta o cwd de volta pra /content no Colab).
cd /content/repo

# Confirma que o pyproject.toml está aqui (sanity check)
if [ ! -f "pyproject.toml" ]; then
    echo "❌ pyproject.toml não encontrado em $(pwd)."
    echo "   A célula 3 (clone) provavelmente falhou. Volte e verifique."
    exit 1
fi

# Instala dependências (uv cria .venv e baixa o que precisa)
# --reinstall força reinstalação se o .venv estiver corrompido
echo "=== uv sync ==="
uv sync --reinstall 2>&1 | tail -15

# Confirma que os entry points foram instalados
echo
echo "=== transcrever --help ==="
uv run transcrever --help 2>&1 | head -5

echo
echo "=== pull-secrets --help ==="
uv run pull-secrets --help 2>&1 | head -5

=== uv sync ===
 + platformdirs==4.11.5
 + protobuf==7.36.1
 + pygments==2.21.0
 + python-dotenv==1.2.3
 + pyyaml==6.0.3
 + referencing==0.37.0
 + rich==15.0.0
 + rpds-py==2026.6.3
 + script-transcricao-videos-31-08-26==0.1.0 (from file:///content/repo)
 + shellingham==1.5.4
 + tokenizers==0.23.1
 + tqdm==4.70.0
 + traitlets==5.16.1
 + typer==0.27.2
 + typing-extensions==4.16.0

=== transcrever --help ===
                                                                                
 Usage: transcrever [OPTIONS] {video}                                           
                                                                                
 Transcreve um vídeo com fidelidade de termos técnicos e nomes próprios.        
                                                                                

=== pull-secrets --help ===
usage: pull-secrets [-h] [--origem {auto,colab,env,manual}]
                    [--env-path ENV_PATH] [--dry-run] [--force]

Sincroniza Secrets do Colab para

## 5. Configurar chave da OpenRouter (a partir dos Secrets do Colab)

As chaves são lidas dos **Secrets** do Colab (ícone 🔑 na barra lateral esquerda), **não** são coladas no notebook. Isso evita expor a chave em logs, screenshots ou ao compartilhar o notebook.

**Antes de rodar esta célula, configure os Secrets:**

1. Clique no ícone **🔑 Secrets** na barra lateral esquerda do Colab.
2. Adicione uma chave chamada `OPENROUTER_API_KEY` com o valor da sua chave (formato `sk-or-v1-...`).
3. Adicione (opcional) `OPENROUTER_MODEL` se quiser fixar um modelo diferente do default.

> Os Secrets são por-usuário e por-notebook. Se você renomear o notebook, precisa reconfigurar.

Se a chave não estiver configurada, a transcrição segue **só com o Whisper** (ainda assim muito fiel, especialmente com `--tema` bem definido).

In [6]:
import os
from google.colab import userdata

def _carregar_secret(nome: str, default: str = "") -> str:
    """Lê um Secret do Colab. Retorna `default` se não estiver configurado."""
    try:
        return userdata.get(nome)
    except Exception:
        return default

# Chave da OpenRouter (obrigatória só se for usar a revisão LLM)
os.environ["OPENROUTER_API_KEY"] = _carregar_secret("OPENROUTER_API_KEY")

# Modelo default (pode ser sobrescrito aqui ou via Secret OPENROUTER_MODEL)
os.environ["OPENROUTER_MODEL"] = _carregar_secret(
    "OPENROUTER_MODEL",
    "meta-llama/llama-3.3-70b-instruct:free",
)

if os.environ["OPENROUTER_API_KEY"]:
    chave = os.environ["OPENROUTER_API_KEY"]
    # Mostra só o prefixo pra confirmar que carregou (sem expor a chave)
    print(f"✓ OPENROUTER_API_KEY carregada dos Secrets ({chave[:10]}…{chave[-4:]})")
    print(f"✓ OPENROUTER_MODEL = {os.environ['OPENROUTER_MODEL']}")
else:
    print("⚠ OPENROUTER_API_KEY não encontrada nos Secrets do Colab.")
    print("  → Para usar a etapa de revisão LLM:")
    print("     1. Abra a aba 🔑 Secrets (barra lateral esquerda).")
    print("     2. Adicione 'OPENROUTER_API_KEY' com sua chave (sk-or-v1-…).")
    print("     3. Re-rode esta célula.")
    print("  → Sem chave, a transcrição segue só com o Whisper (--sem-revisao).")

# Também exporta para o subprocesso (célula 9) herdar via env,
# então o `cli.py` consegue lê-la via os.getenv mesmo rodando fora do kernel.
%env OPENROUTER_API_KEY=$OPENROUTER_API_KEY
%env OPENROUTER_MODEL=$OPENROUTER_MODEL

# (Opcional) Sincroniza Secrets do Colab → .env local do projeto.
# Útil se você for rodar `uv run transcrever` localmente com as mesmas chaves.
# Por padrão, chaves já presentes no .env são preservadas; use --force para sobrescrever.
print()
print("=== pull-secrets --dry-run (apenas informativo) ===")
!uv run pull-secrets --dry-run || echo "(nenhum secret encontrado — pule se for usar só o Whisper)"

✓ OPENROUTER_API_KEY carregada dos Secrets (sk-or-v1-d…f330)
✓ OPENROUTER_MODEL = meta-llama/llama-3.3-70b-instruct:free
env: OPENROUTER_API_KEY=$OPENROUTER_API_KEY
env: OPENROUTER_MODEL=$OPENROUTER_MODEL

=== pull-secrets --dry-run (apenas informativo) ===
📥 Sincronizando Secrets → /content/repo/.env
   (dry-run: nada será escrito)

--- Conteúdo que seria escrito ---
  OPENROUTER_API_KEY=$OPENR…_KEY  (novo)
  OPENROUTER_MODEL=$OPENROUTER_MODEL  (novo)
---------------------------------


## 6. Apontar para o vídeo

O caminho recomendado é usar o **Google Drive** (sem custo de upload, sem consumir espaço do Colab). A célula de upload local fica disponível como fallback, mais abaixo.

In [7]:
# 6.A — Caminho recomendado: processar uma LISTA de vídeos a partir de um diretório no Drive
#
# Vantagens:
#  - Sem custo de upload, sem consumir espaço do Colab.
#  - Lista todos os vídeos do diretório para você escolher (ou processar todos).
#  - Reaproveita o mesmo path em várias execuções — só edita o diretório.

from pathlib import Path

DRIVE_VIDEO_DIR = "/content/drive/MyDrive/000_ai_agents_engineering_26/trilha_claude_code_jd/transcricao_videos_26/video_base"
# ↑ edite aqui se mudar de pasta. Use o seu path real.

EXTENSOES_VALIDAS = {".mkv", ".mp4", ".wav", ".avi", ".mov"}

p_dir = Path(DRIVE_VIDEO_DIR)
if not p_dir.exists():
    raise FileNotFoundError(
        f"Diretório não encontrado: {DRIVE_VIDEO_DIR}\n"
        "Verifique se o Drive está montado (seção 6.1) e se o path está correto."
    )

videos = sorted(
    p for p in p_dir.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSOES_VALIDAS
)

if not videos:
    raise FileNotFoundError(
        f"Nenhum vídeo ({' / '.join(sorted(EXTENSOES_VALIDAS))}) em {DRIVE_VIDEO_DIR}"
    )

print(f"Encontrei {len(videos)} arquivo(s) em {DRIVE_VIDEO_DIR}:\n")
for i, v in enumerate(videos, 1):
    size_mb = v.stat().st_size / 1e6
    print(f"  [{i}] {v.name}  ({size_mb:.1f} MB)")

# Opções de seleção (escolha UMA das quatro):
#   a) Processar TODOS os vídeos do diretório:
#        VIDEO_PATHS = [str(v) for v in videos]
#   b) Processar apenas o PRIMEIRO:
#        VIDEO_PATHS = [str(videos[0])]
#   c) Processar um INTERVALO (ex.: do segundo ao quarto, 0-based, exclusivo):
#        VIDEO_PATHS = [str(v) for v in videos[1:4]]
#   d) Seleção ARBITRÁRIA por índices (0-based):
#        VIDEO_PATHS = [str(videos[i]) for i in [0, 2, 5]]

VIDEO_PATHS = [str(videos[0])]  # ← padrão: apenas o primeiro vídeo

print(f"\nSelecionado(s) ({len(VIDEO_PATHS)}):")
for p in VIDEO_PATHS:
    print(f"  - {p}")

Encontrei 1 arquivo(s) em /content/drive/MyDrive/000_ai_agents_engineering_26/trilha_claude_code_jd/transcricao_videos_26/video_base:

  [1] 2026-08-31 02-09-08.mkv  (946.8 MB)

Selecionado(s) (1):
  - /content/drive/MyDrive/000_ai_agents_engineering_26/trilha_claude_code_jd/transcricao_videos_26/video_base/2026-08-31 02-09-08.mkv


In [ ]:
# 6.B — (Opcional) Processar UM ÚNICO vídeo, com path escrito à mão
#
# Use esta célula APENAS se quiser sobrescrever a lista da célula 6.A com um
# path único (ex.: um vídeo que não está no diretório padrão).
# Descomente o bloco abaixo e ajuste o path.

# from pathlib import Path
#
# VIDEO_PATH = "/content/drive/MyDrive/caminho/do/video.mkv"  # ← edite aqui
# p = Path(VIDEO_PATH)
# if not p.exists():
#     raise FileNotFoundError(f"Não encontrei: {VIDEO_PATH}")
# if p.suffix.lower() not in {".mkv", ".mp4", ".wav", ".avi", ".mov"}:
#     raise ValueError(f"Extensão não suportada: {p.suffix}")
# VIDEO_PATHS = [VIDEO_PATH]
# print(f"Vídeo único selecionado: {VIDEO_PATH} ({p.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# 6.C — (Opcional, fallback) Upload local de um arquivo
#
# ⚠ Use esta célula APENAS se o vídeo NÃO estiver no Drive.
# Upload local de ~950MB leva ~3min e consome espaço do Colab.
# O caminho recomendado é a seção 6.A (lista de vídeos no Drive).
#
# Descomente o bloco abaixo para usar.

# from google.colab import files
# import os
#
# os.makedirs("/content/videos", exist_ok=True)
# %cd /content/videos
# uploaded = files.upload()
# uploaded_name = next(iter(uploaded))
# uploaded_path = "/content/videos/" + uploaded_name
# %cd /content
# ext = uploaded_path.split('.')[-1].lower()
# print(f"\nArquivo recebido: {uploaded_path}")
# print(f"Tamanho: {os.path.getsize(uploaded_path) / 1e9:.2f} GB")
# if ext == "wav":
#     print("→ Entrada é .wav: o ffmpeg será pulado no pipeline.")
# else:
#     print("→ Vídeo: o ffmpeg vai extrair o áudio antes de transcrever.")
# VIDEO_PATHS = [uploaded_path]

## 7. Definir tema e glossário

O **tema** é a chave da fidelidade. Escreva um parágrafo curto descrevendo o assunto do vídeo, mencionando nomes próprios e termos técnicos que devem aparecer com grafia correta.

O **glossário** é uma lista de termos com grafia obrigatória. Crie um arquivo `glossario.txt` no seu PC (1 termo por linha) e faça upload abaixo. Se não tiver glossário, deixe como está.

In [8]:
TEMA = "Claude Code - Visão Geral"  # ← edite aqui

# Glossário opcional
from google.colab import files
import os
glossario_path = None
print("Upload do glossario.txt (ou pule esta etapa se não tiver):")
try:
    uploaded = files.upload()
    glossario_path = "/content/" + next(iter(uploaded))
    print(f"\nGlossário: {glossario_path}")
    with open(glossario_path) as f:
        print("\nTermos:")
        print(f.read())
except Exception as e:
    print(f"Sem glossário: {e}")
    glossario_path = None

Upload do glossario.txt (ou pule esta etapa se não tiver):


Saving glossario.txt to glossario.txt

Glossário: /content/glossario.txt
Sem glossário: [Errno 2] No such file or directory: '/content/glossario.txt'


## 8. Configurar execução

Parâmetros do pipeline. `large-v3` na T4 demora ~5-10min por hora de áudio.

In [ ]:
MODELO_WHISPER = "large-v3"  # tiny, base, small, medium, large-v3
DEVICE = "auto"          # auto, cpu ou cuda
COMPUTE_TYPE = "auto"    # auto, int8, float16, float32
SEM_REVISAO = False      # True = pular etapa do LLM
FORCAR = False           # True = reprocessar mesmo com cache

print(f"Config: modelo={MODELO_WHISPER}, device={DEVICE}, compute={COMPUTE_TYPE}")

## 9. Rodar transcrição 🚀

Esta célula leva alguns minutos. Acompanhe o progresso abaixo.

In [ ]:
%cd /content/repo

import subprocess
import time

# Glossário: cria um vazio se não foi feito upload na célula 7
if 'glossario_path' not in globals() or not glossario_path:
    glossario_path = ""
    print("⚠ Sem glossário — seguindo apenas com o tema.")

print(f"\nProcessando {len(VIDEO_PATHS)} vídeo(s)…\n")
inicio_total = time.time()

for i, video_path in enumerate(VIDEO_PATHS, 1):
    print("=" * 70)
    print(f"[{i}/{len(VIDEO_PATHS)}] {video_path}")
    print("=" * 70)

    cmd = [
        "uv", "run", "transcrever", video_path,
        "--tema", TEMA,
        "--modelo-whisper", MODELO_WHISPER,
        "--device", DEVICE,
        "--compute-type", COMPUTE_TYPE,
    ]
    if glossario_path:
        cmd += ["--glossario", glossario_path]
    if SEM_REVISAO:
        cmd += ["--sem-revisao"]
    if FORCAR:
        cmd += ["--forcar"]

    print("Comando:", " ".join(f'"{c}"' if " " in c else c for c in cmd))
    print()
    rc = subprocess.call(cmd)
    print(f"\n→ exit code: {rc}\n")

    if rc != 0:
        print(f"⚠ Vídeo {i} terminou com erro. Continuando para o próximo…")

duracao_total = time.time() - inicio_total
print("=" * 70)
print(f"✓ Lote concluído em {duracao_total/60:.1f} min "
      f"({len(VIDEO_PATHS)} vídeo(s)).")
%cd /content

## 10. Baixar resultados

Os arquivos gerados ficam em `/content/repo/saidas/<nome-do-video>/`. Faça download de tudo zipado.

In [ ]:
import shutil
from pathlib import Path

saidas_dir = Path("/content/repo/saidas")
if not saidas_dir.exists():
    raise RuntimeError("Nenhuma saída encontrada em /content/repo/saidas/")

# Pega só os diretórios correspondentes aos vídeos processados neste lote
nomes = {Path(p).stem for p in VIDEO_PATHS}
saidas = [p for p in saidas_dir.iterdir() if p.is_dir() and p.name in nomes]

if not saidas:
    raise RuntimeError(f"Nenhuma saída encontrada para os vídeos processados em {saidas_dir}")

print(f"Encontrei {len(saidas)} saída(s):\n")
total_size = 0.0
for s in sorted(saidas):
    size = sum(f.stat().st_size for f in s.rglob("*") if f.is_file())
    total_size += size
    print(f"  - {s.name}  ({size / 1e6:.2f} MB)")

# Empacota cada saída em um zip separado
zips = []
for saida in sorted(saidas):
    zip_base = f"/content/{saida.name}"
    zip_path = f"{zip_base}.zip"
    if Path(zip_path).exists():
        Path(zip_path).unlink()
    shutil.make_archive(zip_base, "zip", saida)
    zips.append((saida.name, zip_path, Path(zip_path).stat().st_size / 1e6))

# Empacota também um zip consolidado com todas as saídas
zip_lote = "/content/saidas_lote.zip"
if Path(zip_lote).exists():
    Path(zip_lote).unlink()
shutil.make_archive(zip_lote.replace(".zip", ""), "zip", saidas_dir)
zip_lote_size = Path(zip_lote).stat().st_size / 1e6

print(f"\n✓ Zips gerados:")
for nome, zp, sz in zips:
    print(f"   {nome}.zip  ({sz:.2f} MB)")
print(f"   saidas_lote.zip  ({zip_lote_size:.2f} MB) — contém todas as saídas")

from google.colab import files
print("\nIniciando downloads…")
files.download(zip_lote)
for _, zp, _ in zips:
    files.download(zp)

## 11. Preview do resultado

Mostra as primeiras linhas do `.txt` para você conferir a fidelidade.

In [ ]:
if saida:
    txt = saida / "transcricao.txt"
    if txt.exists():
        texto = txt.read_text(encoding="utf-8")
        print(f"Total: {len(texto)} caracteres, {len(texto.split())} palavras\n")
        print("=" * 70)
        print(texto[:3000])
        if len(texto) > 3000:
            print(f"\n... [+{len(texto) - 3000} caracteres]")
    else:
        print("transcricao.txt não encontrado")
else:
    print("Sem saída para mostrar")